# Biohub Cell Tracking — Pipeline v1.4

3D+time cell detection, tracking, and lineage submission for the Biohub competition.

| Version | Focus |
|---------|-------|
| v1.1 | Dense-cluster peak pass, adaptive thresholds, intensity sampling |
| v1.2 | Rich linking: distance + motion + intensity + neighborhood |
| v1.3 | Improved division detection (midpoint gate, relaxed count gate) |
| v1.4 | Train hyperparameter search (`CFG.run_hyperparameter_search=True`) |
| v2.0 | Learned detector scaffold (`CFG.detector_backend='learned'`) |

Public baseline: **0.607**. This revision targets detection quality, link precision, and division recall.


## Competition Overview

| | |
|---|---|
| Input | `.zarr` volumes `(T,Z,Y,X)`, one timepoint per chunk |
| Output | `submission.csv` with `node` + `edge` rows |
| Voxel spacing | Z=1.625 µm, Y=X=0.40625 µm |
| Match gate | 7 µm physical distance |
| Metric | Edge Jaccard + Division Jaccard |


In [ ]:
from __future__ import annotations

import gc
import json
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import blosc2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage.feature import peak_local_max
from skimage.filters import threshold_otsu

plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})
np.random.seed(42)


## Configuration


In [ ]:
"""Competition configuration and physical scale constants."""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Tuple

import numpy as np

# Physical voxel spacing (Z, Y, X) in micrometers.
SCALE: Tuple[float, float, float] = (1.625, 0.40625, 0.40625)
MATCH_GATE_UM: float = 7.0
PIPELINE_VERSION: str = "1.4"


@dataclass
class Config:
    """Pipeline hyperparameters. Tuned for metric geometry and CPU runtime."""

    # Paths (resolved at runtime on Kaggle or locally).
    data_root: Path | None = None
    train_dir: Path | None = None
    test_dir: Path | None = None
    output_path: Path = field(default_factory=lambda: Path("submission.csv"))

    # Detector backend: "peaks" (classical) or "learned" (v2 stub).
    detector_backend: str = "peaks"

    # --- v1.1 detection ---
    xy_ds: int = 4
    smooth_sigma: float = 1.0
    min_peak_dist: int = 3
    thresh_rel: float = 0.30
    thresh_hi_percentile: float = 99.8
    min_rel_contrast: float = 0.08
    use_dense_cluster_pass: bool = True
    dense_min_peak_dist: int = 2
    dense_thresh_rel_delta: float = -0.04
    use_adaptive_frame_threshold: bool = True
    adaptive_overcount_penalty: float = 0.04
    min_z_hard: int = 4

    refine_radius_z: int = 2
    refine_radius_yx: int = 5
    nms_radius_um: float = 2.65
    nms_dense_radius_um: float = 2.0
    border_z: int = 3
    border_yx: int = 2
    border_keep_quantile: float = 0.80

    max_frame_count_mult: float = 1.70
    max_frame_count_add: int = 45
    max_nodes_per_frame: int = 20_000

    # --- v1.2 linking ---
    max_link_dist_um: float = 11.0
    use_rich_linking: bool = True
    motion_lambda: float = 0.20
    intensity_lambda: float = 0.15
    neighborhood_lambda: float = 0.10
    neighborhood_k: int = 4
    neighborhood_radius_um: float = 15.0

    # --- v1.3 divisions ---
    detect_divisions: bool = True
    div_parent_dist_um: float = 12.0
    div_sister_dist_um: float = 7.5
    div_min_count_gain: int = 0
    div_require_continued: bool = False
    div_use_midpoint_gate: bool = True
    div_midpoint_dist_um: float = 9.0

    prune_isolated_nodes: bool = True

    # --- v1.4 tuning ---
    run_hyperparameter_search: bool = False
    hyperparam_sample_limit: int = 3
    hyperparam_frames: int = 4

    # Runtime / EDA switches.
    submit_mode: bool = True
    eda_sample_limit: int = 4
    calibration_frames: int = 5
    random_state: int = 42

    @property
    def scale_array(self) -> np.ndarray:
        return np.array(SCALE, dtype=np.float64)

    def resolve_paths(self) -> None:
        """Locate competition data directories."""
        candidates = [
            Path("/kaggle/input/competitions/biohub-cell-tracking-during-development"),
            Path("/kaggle/input/biohub-cell-tracking-during-development"),
        ]
        if self.data_root is None:
            self.data_root = next((p for p in candidates if p.is_dir()), None)
        if self.data_root is not None:
            if self.train_dir is None:
                train = self.data_root / "train"
                self.train_dir = train if train.is_dir() else None
            if self.test_dir is None:
                test = self.data_root / "test"
                self.test_dir = test if test.is_dir() else None

        if Path("/kaggle/working").exists():
            self.output_path = Path("/kaggle/working/submission.csv")

    def copy_with(self, **kwargs) -> "Config":
        """Return a shallow copy with selected fields overridden."""
        from dataclasses import replace

        return replace(self, **kwargs)


## Data Loading


In [ ]:
"""Zarr volume and GEFF graph loading."""

from __future__ import annotations

import json
from pathlib import Path
from typing import List, Tuple

import numpy as np

try:
    import blosc2
except ImportError:
    blosc2 = None  # type: ignore

try:
    import zarr
except ImportError:
    zarr = None  # type: ignore


def list_datasets(split_dir: Path) -> List[str]:
    """Return sorted dataset names (without .zarr suffix)."""
    if not split_dir or not split_dir.is_dir():
        return []
    return sorted(p.name[:-5] for p in split_dir.iterdir() if p.name.endswith(".zarr"))


def read_zarr_meta(zarr_path: Path) -> Tuple[Tuple[int, ...], np.dtype]:
    """Read shape and dtype from zarr metadata."""
    meta_path = zarr_path / "0" / "zarr.json"
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing metadata: {meta_path}")
    meta = json.loads(meta_path.read_text())
    return tuple(meta["shape"]), np.dtype(meta["data_type"])


def load_volume(
    zarr_path: Path,
    t: int,
    shape: Tuple[int, ...],
    dtype: np.dtype,
) -> np.ndarray:
    """Load one timepoint as (Z, Y, X)."""
    chunk = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"
    if blosc2 is not None and chunk.exists():
        raw = blosc2.decompress(chunk.read_bytes())
        return np.frombuffer(raw, dtype=dtype).reshape(shape[1:])
    if zarr is not None:
        arr = zarr.open_array(str(zarr_path / "0"), mode="r")
        return np.asarray(arr[t], dtype=dtype)
    raise ImportError("blosc2 or zarr required to read volumes")


def read_estimated_nodes(geff_path: Path) -> float | None:
    """Read estimated_number_of_nodes from GEFF metadata if present."""
    meta_path = geff_path / "zarr.json"
    if not meta_path.exists():
        return None
    try:
        meta = json.loads(meta_path.read_text())
    except json.JSONDecodeError:
        return None

    def _find(obj):
        if isinstance(obj, dict):
            if "estimated_number_of_nodes" in obj:
                return obj["estimated_number_of_nodes"]
            for v in obj.values():
                found = _find(v)
                if found is not None:
                    return found
        elif isinstance(obj, list):
            for item in obj:
                found = _find(item)
                if found is not None:
                    return found
        return None

    val = _find(meta)
    return float(val) if val is not None else None


def load_geff_graph(geff_path: Path):
    """Load node and edge tables from a GEFF directory."""
    if zarr is None or not geff_path.exists():
        return None, None
    try:
        import pandas as pd

        root = zarr.open_group(str(geff_path), mode="r")
        nodes_grp = root["nodes"]
        edges_grp = root["edges"]
        node_ids = np.asarray(nodes_grp["ids"])
        props = nodes_grp["props"]
        nodes = pd.DataFrame(
            {
                "node_id": node_ids,
                "t": np.asarray(props["t"]["values"]),
                "z": np.asarray(props["z"]["values"]),
                "y": np.asarray(props["y"]["values"]),
                "x": np.asarray(props["x"]["values"]),
            }
        )
        edge_ids = np.asarray(edges_grp["ids"])
        edges = pd.DataFrame({"source_id": edge_ids[:, 0], "target_id": edge_ids[:, 1]})
        return nodes, edges
    except Exception:
        return None, None


## Detection (v1.1)


In [ ]:
"""Anisotropy-aware cell centroid detection (v1.1)."""

from __future__ import annotations

from typing import Optional, Tuple

import numpy as np
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.spatial import cKDTree


try:
    from skimage.feature import peak_local_max
    from skimage.filters import threshold_otsu
except ImportError:
    peak_local_max = None
    threshold_otsu = None


def block_mean_xy(vol: np.ndarray, factor: int) -> np.ndarray:
    """Average-pool XY while preserving Z. Makes the grid ~isotropic in microns."""
    z, y, x = vol.shape
    y2, x2 = (y // factor) * factor, (x // factor) * factor
    arr = vol[:, :y2, :x2].astype(np.float32, copy=False)
    return arr.reshape(z, y2 // factor, factor, x2 // factor, factor).mean(axis=(2, 4))


def robust_threshold(
    sm: np.ndarray,
    cfg: Config,
    thresh_rel: Optional[float] = None,
) -> Tuple[float, float, float]:
    """Otsu with a relative-rise floor; optional per-pass threshold override."""
    alpha = cfg.thresh_rel if thresh_rel is None else float(thresh_rel)
    bg = float(np.median(sm))
    hi = float(np.percentile(sm, cfg.thresh_hi_percentile))
    dyn = max(hi - bg, 1e-6)
    rel_thr = bg + alpha * dyn
    try:
        otsu = float(threshold_otsu(sm)) if threshold_otsu else float(np.percentile(sm, 96.0))
    except Exception:
        otsu = float(np.percentile(sm, 96.0))
    return max(otsu, rel_thr), bg, dyn


def adaptive_thresh_rel(cfg: Config, prev_count: Optional[int], n_detected: int) -> float:
    """Tighten threshold when a frame suddenly exceeds the previous count."""
    rel = cfg.thresh_rel
    if not cfg.use_adaptive_frame_threshold or prev_count is None or prev_count < 8:
        return rel
    if n_detected > int(prev_count * cfg.max_frame_count_mult):
        rel += cfg.adaptive_overcount_penalty
    return rel


def _fallback_peaks(sm: np.ndarray, threshold_abs: float, min_distance: int) -> np.ndarray:
    size = 2 * min_distance + 1
    mx = maximum_filter(sm, size=(size, size, size), mode="nearest")
    mask = (sm >= mx) & (sm > threshold_abs)
    coords = np.argwhere(mask)
    if coords.size == 0:
        return coords.astype(np.int32)
    vals = sm[coords[:, 0], coords[:, 1], coords[:, 2]]
    return coords[np.argsort(-vals)].astype(np.int32)


def _find_peaks(sm: np.ndarray, threshold_abs: float, min_distance: int) -> np.ndarray:
    if peak_local_max is not None:
        return peak_local_max(
            sm,
            min_distance=min_distance,
            threshold_abs=threshold_abs,
            exclude_border=False,
        ).astype(np.int32)
    return _fallback_peaks(sm, threshold_abs, min_distance)


def refine_centroid(
    vol: np.ndarray,
    approx_zyx: np.ndarray,
    cfg: Config,
) -> Tuple[np.ndarray, float]:
    """Intensity-weighted centroid in a local neighborhood."""
    zc, yc, xc = [int(round(v)) for v in approx_zyx]
    z0 = max(0, zc - cfg.refine_radius_z)
    z1 = min(vol.shape[0], zc + cfg.refine_radius_z + 1)
    y0 = max(0, yc - cfg.refine_radius_yx)
    y1 = min(vol.shape[1], yc + cfg.refine_radius_yx + 1)
    x0 = max(0, xc - cfg.refine_radius_yx)
    x1 = min(vol.shape[2], xc + cfg.refine_radius_yx + 1)
    crop = vol[z0:z1, y0:y1, x0:x1].astype(np.float32, copy=False)
    if crop.size == 0:
        return approx_zyx.astype(np.float64), 0.0

    bg = float(np.percentile(crop, 20.0))
    weights = np.clip(crop - bg, 0.0, None)
    total = float(weights.sum())
    if total <= 1e-6:
        loc = np.unravel_index(int(np.argmax(crop)), crop.shape)
        return np.array([z0 + loc[0], y0 + loc[1], x0 + loc[2]], dtype=np.float64), float(crop[loc])

    zz, yy, xx = np.indices(crop.shape)
    refined = np.array(
        [
            z0 + float((zz * weights).sum() / total),
            y0 + float((yy * weights).sum() / total),
            x0 + float((xx * weights).sum() / total),
        ],
        dtype=np.float64,
    )
    return refined, float(weights.max())


def sample_intensity(vol: np.ndarray, coords: np.ndarray) -> np.ndarray:
    """Max intensity in a 3x3x3 neighborhood at each centroid."""
    if len(coords) == 0:
        return np.empty((0,), dtype=np.float32)
    out = np.empty(len(coords), dtype=np.float32)
    z_dim, y_dim, x_dim = vol.shape
    for i, (z, y, x) in enumerate(coords):
        zc, yc, xc = int(round(z)), int(round(y)), int(round(x))
        z0, z1 = max(0, zc - 1), min(z_dim, zc + 2)
        y0, y1 = max(0, yc - 1), min(y_dim, yc + 2)
        x0, x1 = max(0, xc - 1), min(x_dim, xc + 2)
        out[i] = float(vol[z0:z1, y0:y1, x0:x1].max())
    return out


def physical_nms(
    coords_vox: np.ndarray,
    scores: np.ndarray,
    radius_um: float,
    scale: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    if len(coords_vox) <= 1:
        return coords_vox, scores
    pts = coords_vox.astype(np.float64) * scale[None, :]
    order = np.argsort(-scores)
    tree = cKDTree(pts)
    suppressed = np.zeros(len(coords_vox), dtype=bool)
    keep = []
    for i in order:
        if suppressed[i]:
            continue
        keep.append(i)
        for j in tree.query_ball_point(pts[i], r=radius_um):
            suppressed[j] = True
    keep = np.array(keep, dtype=np.int64)
    return coords_vox[keep], scores[keep]


def _map_ds_to_full(coords_ds: np.ndarray, xy_ds: int) -> np.ndarray:
    approx = coords_ds.astype(np.float64)
    approx[:, 1] = approx[:, 1] * xy_ds + (xy_ds - 1) / 2.0
    approx[:, 2] = approx[:, 2] * xy_ds + (xy_ds - 1) / 2.0
    return approx


def _merge_peak_sets(
    primary: np.ndarray,
    secondary: np.ndarray,
    sm: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    if len(primary) == 0 and len(secondary) == 0:
        return primary, np.empty((0,), dtype=np.float32)
    if len(primary) == 0:
        coords = secondary
    elif len(secondary) == 0:
        coords = primary
    else:
        coords = np.vstack([primary, secondary])
        coords = np.unique(coords, axis=0)
    scores = sm[coords[:, 0], coords[:, 1], coords[:, 2]].astype(np.float32)
    return coords, scores


def detect_cells(
    vol: np.ndarray,
    cfg: Config,
    prev_count: Optional[int] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Detect cell centroids in one 3D frame.

    Returns integer (z, y, x) coordinates, detector scores, and sampled intensities.
    """
    z_dim, y_dim, x_dim = vol.shape
    ds = block_mean_xy(vol, cfg.xy_ds)
    sm = gaussian_filter(ds, sigma=cfg.smooth_sigma, mode="nearest")

    thresh_rel = adaptive_thresh_rel(cfg, prev_count, n_detected=0)
    threshold_abs, bg, dyn = robust_threshold(sm, cfg, thresh_rel=thresh_rel)

    coords_primary = _find_peaks(sm, threshold_abs, cfg.min_peak_dist)
    coords_dense = np.empty((0, 3), dtype=np.int32)
    if cfg.use_dense_cluster_pass:
        dense_rel = max(0.08, thresh_rel + cfg.dense_thresh_rel_delta)
        dense_thr, _, _ = robust_threshold(sm, cfg, thresh_rel=dense_rel)
        coords_dense = _find_peaks(sm, dense_thr, cfg.dense_min_peak_dist)

    coords_ds, peak_scores = _merge_peak_sets(coords_primary, coords_dense, sm)
    if coords_ds.size == 0:
        flat = np.argpartition(sm.ravel(), -3)[-3:]
        coords_ds = np.array(np.unravel_index(flat, sm.shape)).T.astype(np.int32)
        peak_scores = sm[coords_ds[:, 0], coords_ds[:, 1], coords_ds[:, 2]].astype(np.float32)

    rel_contrast = (peak_scores - bg) / max(dyn, 1e-6)
    keep = rel_contrast >= cfg.min_rel_contrast
    coords_ds, peak_scores = coords_ds[keep], peak_scores[keep]
    if len(coords_ds) == 0:
        return (
            np.empty((0, 3), dtype=np.int32),
            np.empty((0,), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
        )

    approx = _map_ds_to_full(coords_ds, cfg.xy_ds)
    refined, refined_scores = [], []
    for a, s in zip(approx, peak_scores):
        r, rs = refine_centroid(vol, a, cfg)
        refined.append(r)
        refined_scores.append(max(float(s), rs))
    coords = np.vstack(refined)
    scores = np.array(refined_scores, dtype=np.float32)

    # Hard minimum Z for weak detections (reduces bottom-slice artifacts).
    if cfg.min_z_hard > 0 and len(coords):
        z_floor_score = float(np.quantile(scores, 0.85)) if len(scores) > 8 else np.inf
        keep = (coords[:, 0] >= cfg.min_z_hard) | (scores >= z_floor_score)
        coords, scores = coords[keep], scores[keep]

    cz, cy, cx = coords[:, 0], coords[:, 1], coords[:, 2]
    border = (
        (cz <= cfg.border_z)
        | (cz >= z_dim - 1 - cfg.border_z)
        | (cy <= cfg.border_yx)
        | (cy >= y_dim - 1 - cfg.border_yx)
        | (cx <= cfg.border_yx)
        | (cx >= x_dim - 1 - cfg.border_yx)
    )
    floor = float(np.quantile(scores, cfg.border_keep_quantile)) if len(scores) > 8 else -np.inf
    keep = (~border) | (scores >= floor)
    coords, scores = coords[keep], scores[keep]

    coords, scores = physical_nms(coords, scores, cfg.nms_radius_um, cfg.scale_array)
    if cfg.use_dense_cluster_pass and len(coords) > 1:
        coords, scores = physical_nms(coords, scores, cfg.nms_dense_radius_um, cfg.scale_array)

    if prev_count is not None and prev_count >= 8:
        cap = int(prev_count * cfg.max_frame_count_mult + cfg.max_frame_count_add)
        if len(coords) > cap:
            order = np.argsort(-scores)[:cap]
            coords, scores = coords[order], scores[order]

    if len(coords) > cfg.max_nodes_per_frame:
        order = np.argsort(-scores)[: cfg.max_nodes_per_frame]
        coords, scores = coords[order], scores[order]

    coords = np.rint(coords).astype(np.int32)
    coords[:, 0] = np.clip(coords[:, 0], 0, z_dim - 1)
    coords[:, 1] = np.clip(coords[:, 1], 0, y_dim - 1)
    coords[:, 2] = np.clip(coords[:, 2], 0, x_dim - 1)
    intensities = sample_intensity(vol, coords)
    return coords, scores, intensities


## Tracking (v1.2 / v1.3)


In [ ]:
"""Frame-to-frame linking and division detection (v1.2 / v1.3)."""

from __future__ import annotations

from collections import Counter, defaultdict
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree



def neighbor_signature(xyz_phys: np.ndarray, cfg: Config) -> np.ndarray:
    """K-nearest-neighbor offset vectors as a local context descriptor."""
    n = len(xyz_phys)
    if n == 0:
        return np.empty((0, cfg.neighborhood_k * 3), dtype=np.float64)
    k = min(cfg.neighborhood_k, max(n - 1, 0))
    if k == 0:
        return np.zeros((n, cfg.neighborhood_k * 3), dtype=np.float64)
    tree = cKDTree(xyz_phys)
    dists, idx = tree.query(xyz_phys, k=k + 1)
    sig = np.zeros((n, cfg.neighborhood_k * 3), dtype=np.float64)
    for i in range(n):
        nbrs = idx[i, 1:] if np.ndim(idx) > 1 else []
        for j, nb in enumerate(nbrs[: cfg.neighborhood_k]):
            if dists[i, j + 1] <= cfg.neighborhood_radius_um:
                sig[i, j * 3 : (j + 1) * 3] = xyz_phys[nb] - xyz_phys[i]
    return sig


def _intensity_cost(prev_i: np.ndarray, curr_j: np.ndarray) -> np.ndarray:
    if prev_i.size == 0 or curr_j.size == 0:
        return np.zeros((len(prev_i), len(curr_j)), dtype=np.float64)
    p = prev_i[:, None].astype(np.float64)
    c = curr_j[None, :].astype(np.float64)
    denom = np.maximum(np.maximum(p, c), 1.0)
    return np.abs(p - c) / denom


def _signature_cost(prev_sig: np.ndarray, curr_sig: np.ndarray) -> np.ndarray:
    if len(prev_sig) == 0 or len(curr_sig) == 0:
        return np.zeros((len(prev_sig), len(curr_sig)), dtype=np.float64)
    diff = prev_sig[:, None, :] - curr_sig[None, :, :]
    return np.sqrt((diff ** 2).sum(axis=2))


def build_link_cost(
    prev_phys: np.ndarray,
    curr_phys: np.ndarray,
    cfg: Config,
    prev_intensity: Optional[np.ndarray] = None,
    curr_intensity: Optional[np.ndarray] = None,
    prev_velocity: Optional[np.ndarray] = None,
    prev_sig: Optional[np.ndarray] = None,
    curr_sig: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """Physical distance matrix plus optional rich-linking terms."""
    dist = np.sqrt(((prev_phys[:, None, :] - curr_phys[None, :, :]) ** 2).sum(axis=2))
    cost = dist.copy()

    if cfg.use_rich_linking:
        if prev_velocity is not None and len(prev_velocity) == len(prev_phys):
            predicted = prev_phys[:, None, :] + prev_velocity[:, None, :]
            motion_resid = np.sqrt(((curr_phys[None, :, :] - predicted) ** 2).sum(axis=2))
            cost = cost + cfg.motion_lambda * motion_resid
        if prev_intensity is not None and curr_intensity is not None:
            cost = cost + cfg.intensity_lambda * _intensity_cost(prev_intensity, curr_intensity)
        if prev_sig is not None and curr_sig is not None:
            sig_cost = _signature_cost(prev_sig, curr_sig)
            med = float(np.median(sig_cost[sig_cost > 0])) if np.any(sig_cost > 0) else 1.0
            cost = cost + cfg.neighborhood_lambda * (sig_cost / max(med, 1e-6))

    big = 1e6
    cost = np.where(dist <= cfg.max_link_dist_um, cost, big)
    return cost, dist


def link_frames(
    prev_ids: Sequence[int],
    prev_xyz: np.ndarray,
    curr_ids: Sequence[int],
    curr_xyz: np.ndarray,
    cfg: Config,
    prev_intensity: Optional[np.ndarray] = None,
    curr_intensity: Optional[np.ndarray] = None,
    prev_velocity: Optional[np.ndarray] = None,
    next_xyz: Optional[np.ndarray] = None,
) -> List[Tuple[int, int]]:
    """Hungarian assignment with rich costs plus division pass."""
    if len(prev_ids) == 0 or len(curr_ids) == 0:
        return []

    scale = cfg.scale_array
    prev_phys = prev_xyz.astype(np.float64) * scale[None, :]
    curr_phys = curr_xyz.astype(np.float64) * scale[None, :]

    prev_sig = neighbor_signature(prev_phys, cfg) if cfg.use_rich_linking else None
    curr_sig = neighbor_signature(curr_phys, cfg) if cfg.use_rich_linking else None

    cost, dist = build_link_cost(
        prev_phys,
        curr_phys,
        cfg,
        prev_intensity=prev_intensity,
        curr_intensity=curr_intensity,
        prev_velocity=prev_velocity,
        prev_sig=prev_sig,
        curr_sig=curr_sig,
    )

    ri, ci = linear_sum_assignment(cost)
    big = 1e6

    edges: List[Tuple[int, int]] = []
    parent_children: Dict[int, List[int]] = defaultdict(list)
    matched_curr = set()

    for r, c in zip(ri, ci):
        if cost[r, c] < big:
            edges.append((int(prev_ids[r]), int(curr_ids[c])))
            parent_children[int(r)].append(int(c))
            matched_curr.add(int(c))

    allow_div = cfg.detect_divisions
    if allow_div and cfg.div_min_count_gain > 0 and len(curr_ids) - len(prev_ids) < cfg.div_min_count_gain:
        allow_div = False

    if allow_div:
        for c in range(len(curr_ids)):
            if c in matched_curr:
                continue
            best_p, best_score = None, np.inf
            for p in range(len(prev_ids)):
                if dist[p, c] > cfg.div_parent_dist_um or len(parent_children.get(p, [])) != 1:
                    continue
                sister = parent_children[p][0]
                sister_dist = float(np.linalg.norm(curr_phys[c] - curr_phys[sister]))
                if sister_dist > cfg.div_sister_dist_um:
                    continue
                if cfg.div_use_midpoint_gate:
                    midpoint = 0.5 * (curr_phys[c] + curr_phys[sister])
                    mid_dist = float(np.linalg.norm(prev_phys[p] - midpoint))
                    if mid_dist > cfg.div_midpoint_dist_um:
                        continue
                score = float(dist[p, c] + 0.25 * sister_dist)
                if score < best_score:
                    best_p, best_score = p, score
            if best_p is not None:
                edges.append((int(prev_ids[best_p]), int(curr_ids[c])))
                parent_children[best_p].append(c)
                matched_curr.add(c)

    return edges


def prune_isolated_nodes(
    node_rows: List[dict],
    edge_rows: List[dict],
) -> Tuple[List[dict], List[dict], Dict[str, int]]:
    """Remove detections that never participate in an edge."""
    incident = set()
    for e in edge_rows:
        incident.add(int(e["source_id"]))
        incident.add(int(e["target_id"]))
    kept_nodes = [r for r in node_rows if int(r["node_id"]) in incident]
    kept_ids = {int(r["node_id"]) for r in kept_nodes}
    kept_edges = [
        e
        for e in edge_rows
        if int(e["source_id"]) in kept_ids and int(e["target_id"]) in kept_ids
    ]
    return kept_nodes, kept_edges, {
        "removed_isolated": len(node_rows) - len(kept_nodes),
        "kept_nodes": len(kept_nodes),
        "kept_edges": len(kept_edges),
    }


def count_divisions(edge_rows: List[dict]) -> int:
    src_counts = Counter(int(e["source_id"]) for e in edge_rows)
    return sum(1 for c in src_counts.values() if c >= 2)


## Hyperparameter Search (v1.4)


In [ ]:
"""Systematic hyperparameter search on training data (v1.4)."""

from __future__ import annotations

import copy
from dataclasses import fields
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd



def _edge_proxy(
    gt_nodes: pd.DataFrame,
    gt_edges: pd.DataFrame,
    pred_edges: List[Tuple[int, int]],
    pred_id_to_gt: Dict[int, int],
) -> float:
    if not pred_edges or gt_edges is None or len(gt_edges) == 0:
        return 0.0
    gt_set = set(map(tuple, gt_edges[["source_id", "target_id"]].astype(int).to_numpy()))
    tp = 0
    for s, t in pred_edges:
        gs, gt = pred_id_to_gt.get(s), pred_id_to_gt.get(t)
        if gs is not None and gt is not None and (gs, gt) in gt_set:
            tp += 1
    denom = len(pred_edges) + max(len(gt_set) - tp, 0)
    return tp / denom if denom else 0.0


def _match_nodes_to_gt(
    pred_xyz: np.ndarray,
    gt_frame: pd.DataFrame,
    scale: np.ndarray,
    gate_um: float = MATCH_GATE_UM,
) -> Dict[int, int]:
    """Map predicted index -> gt node_id using greedy nearest matching."""
    if len(pred_xyz) == 0 or len(gt_frame) == 0:
        return {}
    gt_xyz = gt_frame[["z", "y", "x"]].to_numpy(dtype=np.float64)
    gt_ids = gt_frame["node_id"].astype(int).to_numpy()
    pred_phys = pred_xyz.astype(np.float64) * scale
    gt_phys = gt_xyz * scale
    mapping: Dict[int, int] = {}
    used_gt = set()
    for i, p in enumerate(pred_phys):
        d = np.sqrt(((gt_phys - p) ** 2).sum(axis=1))
        j = int(np.argmin(d))
        if d[j] <= gate_um and j not in used_gt:
            mapping[i] = int(gt_ids[j])
            used_gt.add(j)
    return mapping


def score_config_on_sample(
    train_dir: Path,
    name: str,
    cfg: Config,
    n_frames: int = 4,
) -> Dict[str, float]:
    """Proxy score on one training sample using sparse GEFF labels."""
    gt_nodes, gt_edges = load_geff_graph(train_dir / f"{name}.geff")
    if gt_nodes is None:
        return {"recall": np.nan, "edge_proxy": np.nan, "divisions": 0.0, "score": np.nan}

    zarr_path = train_dir / f"{name}.zarr"
    shape, dtype = read_zarr_meta(zarr_path)
    scale = cfg.scale_array

    recalls = []
    edge_proxies = []
    divisions = 0
    prev_xyz = np.empty((0, 3))
    prev_int = np.empty((0,))
    prev_vel = None
    prev_ids: List[int] = []
    pred_id = 1
    idx_to_pred_id: Dict[int, int] = {}
    pred_id_to_gt: Dict[int, int] = {}

    for t in range(min(n_frames, shape[0])):
        vol = load_volume(zarr_path, t, shape, dtype)
        coords, _, intens = detect_cells(vol, cfg, prev_count=len(prev_xyz) if t else None)
        gt_t = gt_nodes[gt_nodes["t"] == t]
        if len(gt_t) and len(coords):
            gt_phys = gt_t[["z", "y", "x"]].to_numpy(dtype=np.float64) * scale
            pred_phys = coords.astype(np.float64) * scale
            matched = 0
            for g in gt_phys:
                if np.min(np.sqrt(((pred_phys - g) ** 2).sum(axis=1))) <= MATCH_GATE_UM:
                    matched += 1
            recalls.append(matched / len(gt_t))

        frame_map = _match_nodes_to_gt(coords, gt_t, scale)
        curr_ids = []
        for i in range(len(coords)):
            pid = pred_id
            pred_id += 1
            curr_ids.append(pid)
            idx_to_pred_id[i] = pid
            if i in frame_map:
                pred_id_to_gt[pid] = frame_map[i]

        if t > 0 and len(prev_ids) and len(curr_ids):
            links = link_frames(
                prev_ids,
                prev_xyz,
                curr_ids,
                coords,
                cfg,
                prev_intensity=prev_int,
                curr_intensity=intens,
                prev_velocity=prev_vel,
            )
            edge_proxies.append(_edge_proxy(gt_nodes, gt_edges, links, pred_id_to_gt))
            src_counts: Dict[int, int] = {}
            for s, _ in links:
                src_counts[s] = src_counts.get(s, 0) + 1
            divisions += sum(1 for c in src_counts.values() if c >= 2)

            if len(links):
                vel = {}
                pos = {pid: prev_xyz[i] for i, pid in enumerate(prev_ids)}
                pos.update({pid: coords[i] for i, pid in enumerate(curr_ids)})
                for s, u in links:
                    if s in pos and u in pos:
                        vel[u] = (pos[u] - pos[s]) * scale
                prev_vel = np.asarray(
                    [vel.get(pid, np.zeros(3)) for pid in prev_ids],
                    dtype=np.float64,
                )

        prev_xyz, prev_int, prev_ids = coords, intens, curr_ids

    recall = float(np.mean(recalls)) if recalls else 0.0
    edge_p = float(np.mean(edge_proxies)) if edge_proxies else 0.0
    div_rate = divisions / max(min(n_frames, shape[0]) - 1, 1)
    score = 0.45 * recall + 0.45 * edge_p + 0.10 * min(div_rate, 1.0)
    return {"recall": recall, "edge_proxy": edge_p, "divisions": float(divisions), "score": score}


def hyperparameter_search(
    train_dir: Path,
    cfg: Config,
    sample_limit: int = 3,
    frames: int = 4,
) -> Tuple[Config, pd.DataFrame]:
    """
    Grid search key hyperparameters on training samples.

    Returns best config and a results table.
    """
    names = list_datasets(train_dir)[:sample_limit]
    if not names:
        return cfg, pd.DataFrame()

    grid = {
        "thresh_rel": [0.26, 0.30, 0.34],
        "max_link_dist_um": [10.5, 11.0, 12.0],
        "div_parent_dist_um": [11.0, 12.0],
        "div_sister_dist_um": [7.0, 7.5, 8.0],
    }

    rows: List[dict] = []
    best_cfg = cfg
    best_score = -1.0

    base = {
        "thresh_rel": cfg.thresh_rel,
        "max_link_dist_um": cfg.max_link_dist_um,
        "div_parent_dist_um": cfg.div_parent_dist_um,
        "div_sister_dist_um": cfg.div_sister_dist_um,
    }

    def _iter_grid(d, keys, i=0, cur=None):
        cur = cur or {}
        if i == len(keys):
            yield cur
            return
        k = keys[i]
        for v in d[k]:
            cur[k] = v
            yield from _iter_grid(d, keys, i + 1, cur)

    for params in _iter_grid(grid, list(grid.keys())):
        trial = cfg.copy_with(**params)
        scores = [score_config_on_sample(train_dir, n, trial, frames) for n in names]
        mean_score = float(np.nanmean([s["score"] for s in scores]))
        row = {**params, "mean_score": mean_score}
        row.update({f"{k}_{i}": s[k] for i, s in enumerate(scores) for k in ("recall", "edge_proxy")})
        rows.append(row)
        if mean_score > best_score:
            best_score = mean_score
            best_cfg = trial.copy_with(**params)

    results = pd.DataFrame(rows).sort_values("mean_score", ascending=False)
    return best_cfg, results


## Detector Backends (v2.0)


In [ ]:
"""Detector backends (v2.0 scaffold)."""

from __future__ import annotations

from typing import Optional, Protocol, Tuple

import numpy as np



class CellDetector(Protocol):
    """Interface for swappable detection backends."""

    def detect(
        self,
        vol: np.ndarray,
        cfg: Config,
        prev_count: Optional[int] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Return coords (N,3), scores (N,), intensities (N,)."""
        ...


class PeakDetector:
    """Classical anisotropy-aware peak detector (default, v1.x)."""

    def detect(
        self,
        vol: np.ndarray,
        cfg: Config,
        prev_count: Optional[int] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        return detect_cells(vol, cfg, prev_count=prev_count)


class LearnedDetector:
    """
    Placeholder for v2.0 learned backends (Cellpose, StarDist, 3D U-Net).

    Attach pretrained weights as a Kaggle dataset and implement `predict`
    to return centroid coordinates in (z, y, x) voxel space.
    """

    def __init__(self, weights_path: str | None = None) -> None:
        self.weights_path = weights_path

    def detect(
        self,
        vol: np.ndarray,
        cfg: Config,
        prev_count: Optional[int] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        if self.weights_path is None:
            return detect_cells(vol, cfg, prev_count=prev_count)
        raise NotImplementedError(
            "LearnedDetector is a scaffold. Subclass and load Cellpose/StarDist weights "
            "from a Kaggle dataset, then map instance centroids to (z,y,x)."
        )


def get_detector(cfg: Config) -> CellDetector:
    if cfg.detector_backend == "learned":
        return LearnedDetector()
    return PeakDetector()


## Visualization


In [ ]:
"""Visualization helpers for EDA and sanity checks."""

from __future__ import annotations

from typing import Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np


def mip(vol: np.ndarray, axis: int = 0) -> np.ndarray:
    """Maximum intensity projection along axis (0=Z, 1=Y, 2=X)."""
    return vol.max(axis=axis)


def plot_volume_slices(
    vol: np.ndarray,
    centroids: Optional[np.ndarray] = None,
    title: str = "",
    z_idx: Optional[int] = None,
) -> plt.Figure:
    """Show XY slice at mid-Z plus ZY and ZX projections."""
    if z_idx is None:
        z_idx = vol.shape[0] // 2
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))

    axes[0].imshow(vol[z_idx], cmap="gray", aspect="equal")
    axes[0].set_title(f"XY @ z={z_idx}")
    axes[1].imshow(mip(vol, axis=2), cmap="gray", aspect="auto")
    axes[1].set_title("ZY (max proj)")
    axes[2].imshow(mip(vol, axis=1), cmap="gray", aspect="auto")
    axes[2].set_title("ZX (max proj)")

    if centroids is not None and len(centroids):
        zc, yc, xc = centroids[:, 0], centroids[:, 1], centroids[:, 2]
        on_slice = np.abs(zc - z_idx) <= 1
        axes[0].scatter(xc[on_slice], yc[on_slice], s=12, c="lime", linewidths=0.5, edgecolors="k")

    if title:
        fig.suptitle(title, y=1.02)
    for ax in axes:
        ax.axis("off")
    fig.tight_layout()
    return fig


def plot_frame_counts(counts: Sequence[int], title: str = "Nodes per frame") -> plt.Figure:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(counts, color="#3366AA", linewidth=1.2)
    ax.set_xlabel("frame")
    ax.set_ylabel("detections")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    return fig


def plot_gt_overlay(
    vol: np.ndarray,
    gt_xyz: np.ndarray,
    pred_xyz: Optional[np.ndarray] = None,
    t: int = 0,
    title: str = "",
) -> plt.Figure:
    """Overlay ground-truth and optional predictions on a mid-Z slice."""
    z_mid = int(np.median(gt_xyz[:, 0])) if len(gt_xyz) else vol.shape[0] // 2
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(vol[z_mid], cmap="gray")
    if len(gt_xyz):
        on = np.abs(gt_xyz[:, 0] - z_mid) <= 1
        ax.scatter(gt_xyz[on, 2], gt_xyz[on, 1], s=20, c="cyan", label="GT", alpha=0.8)
    if pred_xyz is not None and len(pred_xyz):
        on = np.abs(pred_xyz[:, 0] - z_mid) <= 1
        ax.scatter(pred_xyz[on, 2], pred_xyz[on, 1], s=14, c="lime", label="pred", alpha=0.7)
    ax.legend(loc="upper right", fontsize=8)
    ax.set_title(title or f"t={t}, z={z_mid}")
    ax.axis("off")
    fig.tight_layout()
    return fig


## Submission Pipeline


In [ ]:
"""End-to-end pipeline and submission writer."""

from __future__ import annotations

import gc
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd



def _node_row(dataset: str, node_id: int, t: int, zyx) -> dict:
    z, y, x = [int(v) for v in zyx]
    return {
        "dataset": dataset,
        "row_type": "node",
        "node_id": int(node_id),
        "t": int(t),
        "z": z,
        "y": y,
        "x": x,
        "source_id": -1,
        "target_id": -1,
    }


def _edge_row(dataset: str, source_id: int, target_id: int) -> dict:
    return {
        "dataset": dataset,
        "row_type": "edge",
        "node_id": -1,
        "t": -1,
        "z": -1,
        "y": -1,
        "x": -1,
        "source_id": int(source_id),
        "target_id": int(target_id),
    }


def calibrate_detection(
    train_dir: Path,
    cfg: Config,
    sample_limit: int = 4,
    frames_per_sample: int = 5,
) -> float:
    """Sweep thresh_rel on training samples to match per-frame density."""
    detector = get_detector(cfg)
    names = list_datasets(train_dir)[:sample_limit]
    if not names:
        return cfg.thresh_rel

    grid = [0.26, 0.30, 0.34, 0.38]
    best_thresh, best_err = cfg.thresh_rel, np.inf

    for thresh in grid:
        cfg.thresh_rel = thresh
        ratios = []
        for name in names:
            est = read_estimated_nodes(train_dir / f"{name}.geff")
            if est is None or est <= 0:
                continue
            zarr_path = train_dir / f"{name}.zarr"
            shape, dtype = read_zarr_meta(zarr_path)
            n_frames = min(frames_per_sample, shape[0])
            n_pred = 0
            for t in range(n_frames):
                vol = load_volume(zarr_path, t, shape, dtype)
                coords, _, _ = detector.detect(vol, cfg)
                n_pred += len(coords)
            avg_per_frame = n_pred / max(n_frames, 1)
            target_per_frame = est / max(shape[0], 1)
            ratios.append(avg_per_frame / max(target_per_frame, 1e-6))
        if ratios:
            ratio = float(np.median(ratios))
            err = (ratio - 1.0) ** 2
            if ratio > 1.0:
                err += 0.75 * (ratio - 1.0) ** 2
            if err < best_err:
                best_err, best_thresh = err, thresh

    cfg.thresh_rel = best_thresh
    return best_thresh


def apply_train_tuning(cfg: Config) -> Tuple[Config, Optional[pd.DataFrame]]:
    """Run v1.4 hyperparameter search when enabled and train data exists."""
    if not cfg.run_hyperparameter_search or cfg.train_dir is None:
        return cfg, None
    print(f"Running hyperparameter search (v{PIPELINE_VERSION})...")
    best, table = hyperparameter_search(
        cfg.train_dir,
        cfg,
        sample_limit=cfg.hyperparam_sample_limit,
        frames=cfg.hyperparam_frames,
    )
    if len(table):
        print(table.head(3).to_string(index=False))
    return best, table


def process_dataset(
    split_dir: Path,
    name: str,
    cfg: Config,
) -> Tuple[List[dict], Dict[str, object]]:
    """Run detection + tracking on one dataset."""
    detector = get_detector(cfg)
    zarr_path = split_dir / f"{name}.zarr"
    shape, dtype = read_zarr_meta(zarr_path)
    n_t = shape[0]

    node_rows: List[dict] = []
    edge_rows: List[dict] = []
    frame_ids: List[List[int]] = []
    frame_centroids: List[np.ndarray] = []
    frame_intensities: List[np.ndarray] = []
    node_id = 1
    prev_count: Optional[int] = None
    frame_counts: List[int] = []
    velocity_by_id: Dict[int, np.ndarray] = {}
    position_by_id: Dict[int, np.ndarray] = {}

    for t in range(n_t):
        vol = load_volume(zarr_path, t, shape, dtype)
        coords, scores, intens = detector.detect(vol, cfg, prev_count=prev_count)
        del vol
        gc.collect()

        if len(coords):
            order = np.lexsort((coords[:, 2], coords[:, 1], coords[:, 0]))
            coords = coords[order]
            intens = intens[order]

        curr_ids = list(range(node_id, node_id + len(coords)))
        node_id += len(coords)
        frame_ids.append(curr_ids)
        frame_centroids.append(coords)
        frame_intensities.append(intens)
        frame_counts.append(len(coords))
        prev_count = len(coords)

        for nid, pt in zip(curr_ids, coords):
            node_rows.append(_node_row(name, nid, t, pt))
            position_by_id[int(nid)] = pt.astype(np.float64)

    for t in range(1, n_t):
        prev_ids = frame_ids[t - 1]
        curr_ids = frame_ids[t]
        prev_xyz = frame_centroids[t - 1]
        curr_xyz = frame_centroids[t]
        prev_int = frame_intensities[t - 1]
        curr_int = frame_intensities[t]
        prev_vel = np.asarray(
            [velocity_by_id.get(pid, np.zeros(3, dtype=np.float64)) for pid in prev_ids],
            dtype=np.float64,
        )

        links = link_frames(
            prev_ids,
            prev_xyz,
            curr_ids,
            curr_xyz,
            cfg,
            prev_intensity=prev_int,
            curr_intensity=curr_int,
            prev_velocity=prev_vel if cfg.use_rich_linking else None,
        )
        for s, u in links:
            edge_rows.append(_edge_row(name, s, u))
            if s in position_by_id and u in position_by_id:
                velocity_by_id[u] = (
                    position_by_id[u] - position_by_id[s]
                ) * cfg.scale_array

    nodes_before = len(node_rows)
    edges_before = len(edge_rows)
    div_before = count_divisions(edge_rows)

    if cfg.prune_isolated_nodes:
        node_rows, edge_rows, prune_stats = prune_isolated_nodes(node_rows, edge_rows)
    else:
        prune_stats = {"removed_isolated": 0}

    stats = {
        "dataset": name,
        "shape": shape,
        "nodes_before_prune": nodes_before,
        "edges_before_prune": edges_before,
        "nodes_after_prune": len(node_rows),
        "edges_after_prune": len(edge_rows),
        "divisions": count_divisions(edge_rows),
        "divisions_before_prune": div_before,
        "removed_isolated": prune_stats.get("removed_isolated", 0),
        "count_min": int(min(frame_counts)) if frame_counts else 0,
        "count_max": int(max(frame_counts)) if frame_counts else 0,
        "count_mean": float(np.mean(frame_counts)) if frame_counts else 0.0,
    }
    return node_rows + edge_rows, stats


def build_submission(cfg: Optional[Config] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Process all test datasets and write submission.csv."""
    cfg = cfg or Config()
    cfg.resolve_paths()
    if cfg.test_dir is None:
        raise FileNotFoundError("Test directory not found")

    if cfg.train_dir is not None:
        cfg, _ = apply_train_tuning(cfg)
        calibrate_detection(cfg.train_dir, cfg, cfg.eda_sample_limit, cfg.calibration_frames)

    datasets = list_datasets(cfg.test_dir)
    all_rows: List[dict] = []
    all_stats = []
    t0 = time.time()

    print(f"Pipeline v{PIPELINE_VERSION} | detector={cfg.detector_backend} | thresh_rel={cfg.thresh_rel:.2f}")

    for i, name in enumerate(datasets, 1):
        rows, stats = process_dataset(cfg.test_dir, name, cfg)
        all_rows.extend(rows)
        all_stats.append(stats)
        elapsed = time.time() - t0
        print(
            f"[{i}/{len(datasets)}] {name}: "
            f"{stats['nodes_after_prune']} nodes, {stats['edges_after_prune']} edges, "
            f"{stats['divisions']} divisions ({elapsed:.0f}s)"
        )

    cols = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
    sub = pd.DataFrame(all_rows)[cols]
    sub.index.name = "id"
    sub.to_csv(cfg.output_path)
    print(f"Wrote {cfg.output_path} ({len(sub):,} rows) in {(time.time()-t0)/60:.1f} min")
    return sub, pd.DataFrame(all_stats)


def validate_submission(sub: pd.DataFrame, test_dir: Path) -> None:
    """Assert submission format and graph consistency."""
    expected = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
    assert list(sub.columns) == expected, f"Wrong columns: {list(sub.columns)}"
    assert len(sub) > 0, "Empty submission"

    datasets = set(list_datasets(test_dir))
    assert datasets.issubset(set(sub["dataset"].unique())), "Missing test datasets"

    nodes = sub[sub.row_type == "node"]
    edges = sub[sub.row_type == "edge"]
    assert (nodes[["node_id", "t", "z", "y", "x"]] >= 0).all().all()
    assert (nodes[["source_id", "target_id"]] == -1).all().all()
    assert (edges[["node_id", "t", "z", "y", "x"]] == -1).all().all()
    assert (edges[["source_id", "target_id"]] >= 0).all().all()

    for ds, grp in sub.groupby("dataset"):
        nset = set(grp.loc[grp.row_type == "node", "node_id"].astype(int))
        e = grp[grp.row_type == "edge"]
        assert e["source_id"].astype(int).isin(nset).all(), f"dangling source in {ds}"
        assert e["target_id"].astype(int).isin(nset).all(), f"dangling target in {ds}"

    print("Validation passed")


def local_recall_proxy(
    gt_nodes: pd.DataFrame,
    pred_xyz: np.ndarray,
    t: int,
    scale: np.ndarray = np.array(SCALE),
    gate_um: float = MATCH_GATE_UM,
) -> float:
    """Fraction of GT nodes at time t matched by a prediction within gate_um."""
    gt_t = gt_nodes[gt_nodes["t"] == t]
    if len(gt_t) == 0 or len(pred_xyz) == 0:
        return np.nan
    gt_phys = gt_t[["z", "y", "x"]].to_numpy(dtype=np.float64) * scale
    pred_phys = pred_xyz.astype(np.float64) * scale
    matched = 0
    for g in gt_phys:
        d = np.sqrt(((pred_phys - g) ** 2).sum(axis=1))
        if np.min(d) <= gate_um:
            matched += 1
    return matched / len(gt_t)


## Path Resolution


In [ ]:
CFG = Config()
CFG.resolve_paths()
# Fast submit: leave run_hyperparameter_search=False (~2 min)
# Offline tuning: CFG.run_hyperparameter_search = True  # adds ~5-10 min on train
print('pipeline:', PIPELINE_VERSION)
print('train:', CFG.train_dir)
print('test:', CFG.test_dir)
print('output:', CFG.output_path)


## EDA


In [ ]:
RUN_EDA = CFG.train_dir is not None
train_names = list_datasets(CFG.train_dir) if RUN_EDA else []

if RUN_EDA:
    print(f'{len(train_names)} training datasets')
    rows = []
    for name in train_names[:CFG.eda_sample_limit]:
        zpath = CFG.train_dir / f'{name}.zarr'
        shape, dtype = read_zarr_meta(zpath)
        est = read_estimated_nodes(CFG.train_dir / f'{name}.geff')
        rows.append({'name': name, 'T': shape[0], 'shape': shape, 'est_nodes': est})
    display(pd.DataFrame(rows))


## Ground Truth Inspection


In [ ]:
if RUN_EDA and train_names:
    sample = train_names[0]
    gt_nodes, gt_edges = load_geff_graph(CFG.train_dir / f'{sample}.geff')
    zpath = CFG.train_dir / f'{sample}.zarr'
    shape, dtype = read_zarr_meta(zpath)
    vol = load_volume(zpath, 0, shape, dtype)
    pred, _, _ = detect_cells(vol, CFG)
    gt0 = gt_nodes[gt_nodes['t'] == 0][['z', 'y', 'x']].to_numpy() if gt_nodes is not None else np.empty((0, 3))
    fig = plot_gt_overlay(vol, gt0, pred, t=0, title=sample)
    plt.show()
    if gt_nodes is not None:
        n_div = int((gt_edges.groupby('source_id').size() >= 2).sum())
        print(f'GT nodes={len(gt_nodes)}, edges={len(gt_edges)}, divisions={n_div}')
        print(f'Frame-0 recall proxy: {local_recall_proxy(gt_nodes, pred, 0):.3f}')


## Inference


In [ ]:
t0 = time.time()
submission, stats_df = build_submission(CFG)
display(stats_df)
print(f'Pipeline wall time: {(time.time()-t0)/60:.1f} min')


## Submission + Validation


In [ ]:
print(submission['row_type'].value_counts())
display(submission.head(8))
submission.to_csv(CFG.output_path)
validate_submission(submission, CFG.test_dir)
print('Saved', CFG.output_path)


## Runtime Notes

- Default submit: ~2 min CPU on the public 4-dataset test split.
- `run_hyperparameter_search=True` sweeps `thresh_rel`, `max_link_dist_um`, division gates on 3 train samples.
- Set `detector_backend='learned'` when Cellpose/StarDist weights are attached as a Kaggle dataset (v2.0).

## Citations

- van der Walt et al., scikit-image, PeerJ 2014
- Kuhn, The Hungarian Method, Naval Research Logistics 1955
